# 313 — Compliance Agent

Demonstrates the `ComplianceAgent` assessing real loan applications against APRA standards.

**Real findings expected:**
- `LOAN-0002`: LVR=95% → breaches APG-223-THR-008 (≥90% requires senior review)
- `LOAN-0013`: LVR=92% → same threshold
- `LOAN-0007`: LVR=80% → borderline, serviceability buffer check required

In [ ]:
%run 311_agent_setup.ipynb

In [ ]:
from src.agent.compliance_agent import ComplianceAgent

agent = ComplianceAgent(tools=TOOLS, execute_tool_fn=execute_tool)

## Example 1: High LVR loan — LOAN-0002 (LVR=95)

In [ ]:
result = agent.run('Is LOAN-0002 compliant with APG-223?')
print(f'Verdict:    {result.verdict}')
print(f'Confidence: {result.confidence}')
print(f'Requirements: {result.requirement_ids}')
print(f'Thresholds breached: {result.threshold_breaches}')
print(f'Assessment ID: {result.assessment_id}')
print(f'\nReasoning steps:')
for i, step in enumerate(result.reasoning_steps, 1):
    print(f'  {i}. {step}')
print(f'\nCypher queries used: {len(result.cypher_used)}')
for q in result.cypher_used:
    print(f'  --- {q[:120]}...')

## Example 2: Near-threshold loan — LOAN-0007 (LVR=80)

In [ ]:
result2 = agent.run(
    'Assess LOAN-0007 against APG-223 serviceability requirements. '
    'The borrower has no declared non-salary income.'
)
print(f'Verdict:    {result2.verdict}')
print(f'Confidence: {result2.confidence}')
print(f'Thresholds breached: {result2.threshold_breaches}')

## Example 3: Regulation scoping — what applies to residential secured loans?

In [ ]:
result3 = agent.run(
    'Which APRA quantitative thresholds apply to residential secured loans '
    'in the Australian federal jurisdiction? List them with values.'
)
print(f'Verdict: {result3.verdict}')
print(f'Requirements: {result3.requirement_ids}')

## Layer 3 verification

Confirm Assessment nodes were written to the graph.

In [ ]:
assessments = execute_tool('read-neo4j-cypher', {
    'query': """
        MATCH (a:Assessment)
        OPTIONAL MATCH (a)-[:HAS_FINDING]->(f:Finding)
        RETURN a.assessment_id AS id, a.verdict AS verdict,
               a.confidence AS confidence, count(f) AS finding_count
        ORDER BY a.created_at DESC
        LIMIT 10
    """
})
import json
print(json.dumps(assessments, indent=2, default=str))